# Lab 12: Predictive Modeling — The Same Machinery, A New Scorecard
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 75 min Core + 30 min Write it yourself + 20 min Extension + take-home Challenge

---

**Learning Objectives:**
- Fit OLS by least squares and recover the identical coefficients three ways
- Explain why R-squared cannot fall when a predictor is added, and what that costs you
- Split data before fitting, and report error on observations the model never saw
- Distinguish the two jobs regression does — estimating a parameter and predicting an outcome
- Write a small function that scores any set of predictors on the same held-out rows

**Dataset:** California Housing (scikit-learn, 20,640 districts)

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one.

**How this lab works:** Parts 1-3 are GUIDED: run each cell as it is, then answer the questions. Part 4 has four one-line blanks in two cells. "Write it yourself" is required: a short function you write from a blank cell. The Extension is optional, and the Challenge is a take-home.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI coding section: you write every line yourself. The one use of AI is drafting your README from the prompt at the end.

---

> ### What you write, and what you run
>
> **You write:**
> - **Part 4, Steps 7 and 8:** four one-line blanks (`____`). In Step 7, the coefficient on `AveRooms` and its 95% interval, read by name. In Step 8, a model fitted on the training rows and its test RMSE, modelled on Step 6.
> - **Write it yourself (required),** after Part 4: `holdout_rmse()`, a function that splits a table the way Step 6 does, fits on the training rows and gives back the test RMSE, then one call to it with median income alone and a print. About seven lines from a blank cell. The check cell below it must print "Passed". Then one or two sentences of text comparing the result with Step 6.
> - The answers to the interpretation questions after each part, in text cells.
>
> **You run and read:** everything else. Setup and Parts 1-3 are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The Extension (optional) loads two FRED series for you, then has four one-line blanks and a three-sentence answer.
>
> **The Challenge** (take-home) has no starter code: you write it from the specification, in new cells. The **README** is text only: the prompt at the end is the one place you use AI, and only for documentation.

## Setup

Step 4 draws random numbers from one generator, `rng`, and each draw moves it on. So your numbers
match the expected output only if you run the cells once, from the top, in order. If you run a cell
twice, use **Runtime → Restart session** and run from the top again.

In [ ]:
# GUIDED — run as-is
# Step 1: imports and the random generator
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from pathlib import Path

rng = np.random.default_rng(seed=42)    # the noise columns in Step 4 come from rng, in order
RANDOM_STATE = 42                       # fixes which rows land in the test set in Steps 6 and 8

# savefig will not create a folder, and a fresh Colab session has no figures/ folder
Path("figures").mkdir(exist_ok=True)

print("Setup complete.")

## Part 1: One Predictor, Three Routes to the Same Line (GUIDED — 15 min)

Chapter 12 makes a claim worth checking rather than believing: the closed-form
solution, the numerical optimiser and the library all return the same
coefficients, because they are solving the same problem.

Each row of California Housing is a district (a census block group) in the 1990 census. `MedInc`
is the district's median household income, in tens of thousands of dollars; `value`, the
outcome, is its median house value, in hundreds of thousands of dollars.

In [ ]:
# GUIDED — run as-is
# Step 2: load California Housing, one row per district
data = fetch_california_housing(as_frame=True)
# as_frame=True gives a DataFrame; its outcome column, MedHouseVal, is renamed to the shorter "value"
df = data.frame.rename(columns={"MedHouseVal": "value"})

print(f"Shape: {df.shape}")
print(df[["MedInc", "AveRooms", "HouseAge", "value"]].describe().round(2))

**Expected output:** 20,640 rows and 9 columns. Median income runs from 0.50 to 15.00, and `value`
from 0.15 to 5.00. Keep that maximum of 5.00 in mind: Part 2 comes back to it.

Step 3 fits one line three ways. The closed form is Chapter 12's slope formula, the covariance of
x and y over the variance of x. `np.cov(x, y)` gives a 2 x 2 table, and `[0, 1]` picks the
covariance between `x` and `y`; `ddof=1` divides by n − 1 in both the covariance and the variance.
`sm.OLS(y, X).fit()` is Lab 10's regression: the outcome first, then the regressors, with
`sm.add_constant` adding the column of 1s for the intercept. scikit-learn is new:
`LinearRegression()` makes an empty model; `.fit(X, y)` fits it and hands it back. The order is the
reverse of statsmodels: predictors first, then the outcome. `LinearRegression` adds its own
intercept, but it wants the predictors as a table: `x.reshape(-1, 1)` turns the flat
array `x` into a table with one column. `.values` takes the plain NumPy array out of a DataFrame
column.

In [ ]:
# GUIDED — run as-is
# Step 3: the same line, three ways
x = df["MedInc"].values
y = df["value"].values

# Route 1, the closed form: slope = cov(x, y) / var(x), and the line passes through the two means
b1 = np.cov(x, y, ddof=1)[0, 1] / np.var(x, ddof=1)
b0 = y.mean() - b1 * x.mean()

# Route 2, statsmodels: params[0] is the intercept, params[1] the slope
sm_fit = sm.OLS(y, sm.add_constant(x)).fit()

# Route 3, scikit-learn, which marks what it learned with a trailing _: intercept_ and coef_
sk_fit = LinearRegression().fit(x.reshape(-1, 1), y)

# <24 pads the text to 24 characters; >12.6f right-aligns a number in 12, with 6 decimals
print(f"{'route':<24}{'intercept':>12}{'slope':>12}")
print("-" * 48)
print(f"{'closed form':<24}{b0:>12.6f}{b1:>12.6f}")
print(f"{'statsmodels':<24}{sm_fit.params[0]:>12.6f}{sm_fit.params[1]:>12.6f}")
print(f"{'scikit-learn':<24}{sk_fit.intercept_:>12.6f}{sk_fit.coef_[0]:>12.6f}")

# assert stops with an error if its condition is False; atol=1e-6 asks for agreement to six decimals
assert np.allclose([b0, b1], sm_fit.params, atol=1e-6)
assert np.allclose([b0, b1], [sk_fit.intercept_, sk_fit.coef_[0]], atol=1e-6)

**Expected output:** all three rows read intercept 0.450856 and slope 0.417938, and the two asserts
pass without a word. The three methods compute the same line.

### Interpretation Questions

1. Interpret the slope in the units of the data: a one-unit rise in median income is associated with what change in median house value?
2. The intercept is the fitted value at `MedInc = 0`. Is that a meaningful quantity here? Say what it is doing in the model regardless.
3. Chapter 12 says the same machinery does two different jobs. State the two, and say which one this cell is doing.

*Your answers here:*

1. 
2. 
3. 

## Part 2: R-Squared, and Why It Cannot Fall (GUIDED — 15 min)

R-squared is the number most often quoted for a regression, and it is easy to push up without
making the model any better.

Step 4 puts k columns of random numbers, which have nothing to do with house value, beside median
income, refits the regression, and prints R-squared and adjusted R-squared. Each k gets fresh noise.
`np.column_stack(columns)` sets the arrays in the list side by side as columns, as in Lab 10.

In [ ]:
# GUIDED — run as-is
# Step 4: add pure noise and watch R-squared
print(f"{'noise columns':>14}{'R-squared':>12}{'adjusted':>12}")
print("-" * 40)

for k in [0, 1, 5, 20, 100]:
    columns = [x]                                   # median income first
    for j in range(k):
        columns.append(rng.normal(0, 1, len(y)))    # one column of random numbers per pass
    fit = sm.OLS(y, sm.add_constant(np.column_stack(columns))).fit()
    print(f"{k:>14}{fit.rsquared:>12.4f}{fit.rsquared_adj:>12.4f}")

**Expected output:** R-squared climbs at every row, from 0.4734 with no noise to 0.4758 with 100
columns of it. Adjusted R-squared reads 0.4734, 0.4734, 0.4735, 0.4734 and 0.4732: up at 5 columns,
down by 100. Each row draws fresh noise, so the rows are not one model growing a column at a time.
What is guaranteed is that each row's R-squared is at least the first row's, because every row
still contains median income.

Step 5 draws Step 3's line through the data, and then plots each district's residual (actual value
minus fitted value) against its fitted value. It reuses `sm_fit` from Step 3.

In [ ]:
# GUIDED — run as-is
# Step 5: the fitted line, and what it leaves over
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(x, y, s=2, alpha=0.15, color="darkslategray")
xs = np.linspace(x.min(), x.max(), 100)        # 100 evenly spaced incomes to draw the line through
axes[0].plot(xs, sm_fit.params[0] + sm_fit.params[1] * xs, color="firebrick", lw=2)
axes[0].set_xlabel("Median income")
axes[0].set_ylabel("Median house value")
axes[0].set_title(f"R-squared = {sm_fit.rsquared:.3f}")

axes[1].scatter(sm_fit.fittedvalues, sm_fit.resid, s=2, alpha=0.15, color="darkslategray")
axes[1].axhline(0, color="firebrick", lw=1)
axes[1].set_xlabel("Fitted value")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs fitted")

plt.tight_layout()
plt.savefig("figures/ch12_fit_and_residuals.png", dpi=150, bbox_inches="tight")
plt.show()

**What to look for:** the left panel's title, R-squared = 0.473, is Step 4's first row. Then look
at the straight diagonal edge across the top of the right-hand cloud (the flat row at 5 on the
left), and at how wide the right-hand cloud is at small fitted values compared with large ones.
Question 3 asks what each one means.

### Interpretation Questions

1. R-squared rose from adding random numbers. Explain the mechanism in one sentence, and say what this implies about choosing between models by R-squared.
2. Adjusted R-squared eventually falls. What is it charging for, and why is it still not a safe model-selection rule?
3. The right-hand panel shows two problems the left-hand number cannot see. Name both.

*Your answers here:*

1. 
2. 
3. 

## Part 3: The Test Set (GUIDED — 20 min)

Everything so far scored the model on the data it was fitted to. That answers
"how well does this line describe what I already have", which is not the question
a prediction is for.

`train_test_split` shuffles the rows and deals them into a training set and a test set:
`test_size=0.2` puts one row in five in the test set, and `random_state` fixes the shuffle, so
everyone gets the same split. It gives back four pieces, in the order X_train, X_test, y_train,
y_test. `.predict()` gives the model's fitted value for each row it is handed.
`mean_squared_error(actual, predicted)` is the average squared error, and its square root, the
RMSE, is in the units of the outcome. `r2_score` is R-squared, computed on whichever rows you give it.

In [ ]:
# GUIDED — run as-is
# Step 6: split FIRST, then fit on the training rows only. The order is the whole point
features = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
            "Population", "AveOccup", "Latitude", "Longitude"]
X_train, X_test, y_train, y_test = train_test_split(
    df[features], df["value"], test_size=0.2, random_state=RANDOM_STATE)

model = LinearRegression().fit(X_train, y_train)

# Score the same model twice: on the rows it was fitted to, and on rows it never saw
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

# :, adds a thousands comma
print(f" train: RMSE = {train_rmse:.4f}   R-squared = {r2_score(y_train, train_pred):.4f}"
      f"   n = {len(y_train):,}")
print(f"  test: RMSE = {test_rmse:.4f}   R-squared = {r2_score(y_test, test_pred):.4f}"
      f"   n = {len(y_test):,}")

**Expected output:** on the 16,512 training rows, RMSE 0.7197 and R-squared 0.6126; on the 4,128
test rows, RMSE 0.7456 and R-squared 0.5758. This model uses all eight features, so it is a
different model from Steps 3-5's median income alone.

### Interpretation Questions

1. Why must the split happen before any fitting, scaling or imputation? Name the specific thing that leaks if it does not.
2. The train and test errors here are close. What would a large gap have told you, and what would a test error *below* the train error suggest?
3. RMSE is in the units of the outcome. State this model's typical error in dollars, and say whether you would price a house with it.

*Your answers here:*

1. 
2. 
3. 

## Part 4: The Two Jobs (YOUR TASK — 25 min)

Chapter 12's central claim: the algebra is identical whether you want a
coefficient or a prediction, but the scorecard is not. Steps 7 and 8 use the same specification,
value on median income and rooms per household, for the two jobs.

**Reading results by name.** In Lab 10's Extension the regression was fitted on plain arrays, so
`conf_int()` came back as a NumPy array and `conf_int()[1]` picked row 1 by position. Step 7's
`est` is fitted on a DataFrame, `sm.add_constant(df[[...]])`, so its `.params` and `.conf_int()` come
back labelled with the column names, and you pick a row by name with `.loc[...]`, as Lab 10's
Step 10 read `params["treat"]`. `conf_int()` has one row per coefficient and two columns, the lower
and the upper limit, so `lo, hi = ` followed by one row unpacks that row into two names.

In [ ]:
# YOUR TASK — fill in the two blanks (____)
# Step 7: the estimation job. How is rooms per household associated with value, holding income fixed?
est = sm.OLS(df["value"], sm.add_constant(df[["MedInc", "AveRooms"]])).fit()

# Blank 1: the coefficient on AveRooms. est was fitted on a DataFrame, so read it by name
coef = ____
# Blank 2: the coefficient's 95% interval, as two names: the lower and the upper limit
lo, hi = ____

# +.4f always prints the sign
print(f"Estimation:  AveRooms coefficient {coef:+.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

**Expected output:** a coefficient of −0.0381, with a 95% interval of [−0.0430, −0.0333].
*NameError: name '____' is not defined* means a blank is still empty. *too many values to unpack*
means Blank 2 handed `lo, hi` a whole column of the table, not the one row you need. If the
interval reads [0.0000, 1.0000], you unpacked the whole table.

In [ ]:
# YOUR TASK — fill in the two blanks (____)
# Step 8: the prediction job. How well does the same specification predict rows it never saw?
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    df[["MedInc", "AveRooms"]], df["value"], test_size=0.2, random_state=RANDOM_STATE)

# Blank 1: a LinearRegression fitted on the training rows only, as in Step 6
pred_fit = ____
# Blank 2: that model's RMSE on the test rows, as in Step 6
pred_rmse = ____

print(f"Prediction:  test RMSE {pred_rmse:.4f}")

**Expected output:** a test RMSE of 0.8379. If you get 0.8313, you scored the training rows.
0.8378 means you fitted on every row, test rows included; 0.7022 means you left out the square root.

### Interpretation Questions

1. Suppose you added twenty columns of pure noise to this specification, as Step 4 did. What would happen to its in-sample R-squared, and what would happen to its test RMSE? Explain why the two need not move together.
2. A colleague says "the model has a significant coefficient, so it should predict well." Give a counterexample.
3. Name the scorecard Step 7 used and the one Step 8 used, and give a result that would pass one and fail the other.

*Your answers here:*

1. 
2. 
3. 

---

## Write it yourself (required — 30 min)

This part is required, and it is short. The check cell below it must print "Passed" before you
submit.

**The task.** Step 6 scored eight predictors on the test rows, and Step 8 scored two. How well
does median income do on its own? Rather than copy the split, fit and score lines a third time,
turn them into a function.

Write a function `holdout_rmse(data, features)` that splits `data[features]` and `data["value"]`
the way Step 6 does (`test_size=0.2`, `random_state=RANDOM_STATE`), fits a `LinearRegression` on
the training rows, and gives back the RMSE on the test rows, as one number. With Step 6's list,
`holdout_rmse(df, features)` gives Step 6's test RMSE. Then call it with median income alone,
`["MedInc"]`, store the result as `rmse_income_only`, and print it to 4 decimals.

- It takes about seven lines: the `def` line, a one-line docstring, the split, the fit, the
  `return`, the call and a print.
- Step 8 has the body you need. Inside the function, use the two arguments, `data` and
  `features`, wherever Step 8 names its table and its columns.
- Give back the RMSE alone: not the model, and not the squared error.
- Lab 10's `coef_on_d()` shows the shape of a function: the `def` line, a one-line docstring, the
  body, and `return`.

Then run the check cell. It calls your function with Step 6's features and compares the answer
with Step 6's; it also tries it on changed copies of the table and on a different list of
features. Then it checks `rmse_income_only`. `assert` stops with its message when its condition is
False; the message tells you what to fix.

**Last, in the text cell under the check:** in one or two sentences, how much larger is the test
error with median income alone than with all eight features, in dollars (`value` is in
\$100,000s)? And why is this a fair way to choose between the two models, when Part 2 showed that
comparing their in-sample R-squared is not?

In [ ]:
# WRITE IT YOURSELF (required): holdout_rmse(), then rmse_income_only for median income alone

In [ ]:
# CHECK — for "Write it yourself" above

# globals() holds every name defined so far in this notebook
if "holdout_rmse" not in globals():
    print("Not written yet: write holdout_rmse() in the cell above and run it, then run this check.")
else:
    # try/except catches an error inside your function, so the message can say where to look
    try:
        yours = holdout_rmse(df, features)
    except Exception as e:
        hint = "Read the error above: it says what went wrong."
        if "'squared'" in str(e):          # the keyword squared=False, removed from scikit-learn
            hint = ("mean_squared_error no longer takes squared=False: take np.sqrt of it, "
                    "as Step 6 does.")
        elif "inconsistent numbers of samples" in str(e):
            hint = ("train_test_split gives back X_train, X_test, y_train, y_test, in that order, "
                    "and .fit() takes the predictors first, then the outcome.")
        raise AssertionError(
            f"holdout_rmse(df, features) stopped with an error:\n  {e}\n{hint}") from None
    assert yours is not None, "holdout_rmse gave back nothing. Does it end with a return line?"
    # np.ndim is 0 for a single number
    assert np.ndim(yours) == 0 and isinstance(yours, (int, float, np.floating)), (
        "holdout_rmse gives back something other than one number (a model, a list, or text). "
        "Give back the test RMSE alone.")
    assert not np.isclose(yours, train_rmse), (
        "That is Step 6's training RMSE. Score the test rows, the ones the model never saw.")
    assert not np.isclose(yours, test_rmse ** 2), (
        "That is the mean squared error. The RMSE is its square root.")
    assert not np.isclose(yours, r2_score(y_test, test_pred)), (
        "That is the test R-squared. Give back the RMSE.")
    assert not np.isclose(yours, np.mean(np.abs(y_test - test_pred))), (
        "That is the mean absolute error. Give back the RMSE: the square root of the mean squared error.")
    assert np.isclose(holdout_rmse(df, features), yours), (
        "Two calls with the same arguments give different answers, so the split changes every "
        "time. Fix it with random_state=RANDOM_STATE, as Step 6 does.")

    # Changed copies of the table: value doubled, MedInc flattened, and value moved up by 100
    doubled = df.copy()
    doubled["value"] = 2 * df["value"]
    flat = df.copy()
    flat["MedInc"] = 1.0
    shifted = df.copy()
    shifted["value"] = df["value"] + 100
    assert not np.isclose(holdout_rmse(doubled, features), yours), (
        "Doubling every value in the table you pass in leaves your answer unchanged. Does the "
        "function take the outcome from its data argument, or from the notebook's df?")
    assert not np.isclose(holdout_rmse(flat, features), yours), (
        "Setting MedInc to 1.0 in the table you pass in leaves your answer unchanged. Does the "
        "function take the predictors from its data argument, or from the notebook's df?")
    assert np.isclose(holdout_rmse(shifted, features), yours), (
        "Adding 100 to every value changes your RMSE, so the model has no intercept. "
        "LinearRegression() fits one for you; with sm.OLS, add one with sm.add_constant.")
    assert not np.isclose(holdout_rmse(df, ["MedInc"]), yours), (
        "holdout_rmse(df, ['MedInc']) gives the same answer as all eight features. Does the "
        "function use its features argument, or the notebook's features list?")

    # Two models that have seen the test rows: one fitted on every row, one on the test rows alone
    leaky = LinearRegression().fit(df[features], df["value"])
    assert not np.isclose(yours, np.sqrt(mean_squared_error(y_test, leaky.predict(X_test)))), (
        "Your model has seen the test rows: it was fitted on every row before scoring. Fit on "
        "the training rows only.")
    on_test = LinearRegression().fit(X_test, y_test)
    assert not np.isclose(yours, np.sqrt(mean_squared_error(y_test, on_test.predict(X_test)))), (
        "Your model was fitted on the test rows. Fit on the training rows, then score the test rows.")
    # round(v, places) keeps that many decimals; try 2 to 5 of them
    for places in [2, 3, 4, 5]:
        assert not (np.isclose(yours, round(test_rmse, places), rtol=0, atol=1e-9)
                    and not np.isclose(yours, test_rmse, rtol=0, atol=1e-9)), (
            "Your number is rounded. Give back the full number; round only when you print.")
    assert np.isclose(yours, test_rmse), (
        f"holdout_rmse(df, features) gives {yours:.4f}, and Step 6 printed {test_rmse:.4f}. "
        "Split the way Step 6 does: test_size=0.2 and random_state=RANDOM_STATE.")

    two = holdout_rmse(df, ["MedInc", "AveRooms"])
    if "pred_rmse" in globals() and np.ndim(pred_rmse) == 0:
        assert np.isclose(two, pred_rmse), (
            f"Your function is right, and for MedInc and AveRooms it gives {two:.4f}, but Step 8's "
            f"pred_rmse is {pred_rmse:.4f}. Go back and fix Step 8.")

    assert "rmse_income_only" in globals(), (
        "Your function works. Now call it with median income alone and store the result as "
        "rmse_income_only.")
    mine = rmse_income_only
    one = holdout_rmse(df, ["MedInc"])
    assert np.ndim(mine) == 0, "rmse_income_only should be one number."
    assert not np.isclose(mine, test_rmse), (
        "rmse_income_only is the eight-feature RMSE from Step 6. Use median income alone.")
    assert not np.isclose(mine, two), (
        "rmse_income_only uses MedInc and AveRooms, Step 8's pair. Use median income alone.")
    for places in [2, 3, 4, 5]:
        assert not (np.isclose(mine, round(one, places), rtol=0, atol=1e-9)
                    and not np.isclose(mine, one, rtol=0, atol=1e-9)), (
            "rmse_income_only is rounded. Store the full number; round only when you print.")
    assert np.isclose(mine, one), (
        f"rmse_income_only is {mine:.4f}, but holdout_rmse(df, ['MedInc']) gives {one:.4f}.")
    print("Passed. holdout_rmse matches Step 6, and uses the table and the features it is given.")
    print(f"Test RMSE, median income alone: {mine:.4f}   (all eight features, Step 6: {test_rmse:.4f})")

*Your answer here (the gap in dollars, and why a test-set comparison is fair where in-sample R-squared was not):*

---

## Extension (Optional — 20 min): Okun's Law

This is regression the way economists most often use it: the coefficient itself is the result you
want. Okun's Law relates the
change in unemployment to real GDP growth:

    change in unemployment  =  a  +  b * GDP growth

b is the quantity of interest. Nobody is trying to predict next quarter's unemployment with this
regression; they want to know the number itself.

Step 9 is done for you. It pulls two FRED series from the public CSV endpoint, with no API key, the
same route Lab 3 used: GDPC1, real GDP, quarterly, in billions of chained 2017 dollars, and UNRATE,
the civilian unemployment rate, monthly, in percent. `.resample("QE").mean()` averages the three
months of each quarter, and `.to_period("Q")` labels each value by its quarter, so the two series
line up row for row. `pct_change()` and `diff()` turn levels into changes from the quarter before.

In [ ]:
# EXTENSION — run as-is
# Step 9: two FRED series, made quarterly and turned into changes
def load_fred(series_id, start="1948-01-01"):
    """One FRED series as a Series indexed by date. No API key needed."""
    url = (f"https://fred.stlouisfed.org/graph/fredgraph.csv?"
           f"id={series_id}&cosd={start}")
    frame = pd.read_csv(url)
    frame.columns = ["date", series_id]
    frame["date"] = pd.to_datetime(frame["date"])
    # errors="coerce" turns anything that is not a number into NaN, which dropna then removes
    frame[series_id] = pd.to_numeric(frame[series_id], errors="coerce")
    return frame.set_index("date")[series_id].dropna()


gdp = load_fred("GDPC1").to_period("Q")                           # already quarterly
urate = load_fred("UNRATE").resample("QE").mean().to_period("Q")  # monthly -> quarterly average

# each key becomes a column; rows are matched by quarter
okun = pd.DataFrame({"gdp": gdp, "u": urate}).dropna()
okun["gdp_growth"] = okun["gdp"].pct_change() * 100    # % change, quarter on quarter
okun["d_unrate"] = okun["u"].diff()                    # percentage-point change
okun = okun.dropna()                                   # the first quarter has no change to compute

print(f"{len(okun)} quarters, {okun.index.min()} to {okun.index.max()}")

In [ ]:
# EXTENSION (optional) — fill in the four blanks (____)
# Step 10: estimate Okun's coefficient

# Blank 1: the regressors, a constant plus gdp_growth, kept as a DataFrame (double brackets)
X_okun = ____

# Blank 2: OLS of d_unrate on X_okun, outcome first, as in Step 7
okun_fit = ____

# Blanks 3 and 4: the coefficient on gdp_growth and its 95% interval, read by name as in Step 7
b = ____
lo_b, hi_b = ____

# nobs, the number of observations, is stored as a decimal; int() drops the .0
print(f"Okun coefficient b = {b:.3f}   95% CI [{lo_b:.3f}, {hi_b:.3f}]")
print(f"R-squared: {okun_fit.rsquared:.3f}   n = {int(okun_fit.nobs)} quarters")

**Expected output:** run on 4 October 2026, Step 9 found 313 quarters, 1948Q2 to 2026Q2, and Step 10
gives b = −0.470, a 95% interval of [−0.516, −0.424], and an R-squared of 0.566. FRED adds a
quarter every three months and revises old ones, so your last decimal can differ. A b of −0.47
reads: each extra percentage point of quarterly real GDP growth goes with a fall of about 0.47
points in the unemployment rate.

**Your answer.** R-squared here comes out around 0.5-0.6, but this regression is not judged by its
fit. In
three sentences, explain to someone who only knows prediction (a) what b is for, (b) why a version
of this same regression with an R-squared of 0.2 would still be worth reporting, and (c) what would
have to be true of the CI before you stopped trusting b.

*Your answer here:*

---

## Challenge (Take-Home): Where the Line Breaks

The California Housing outcome is censored at 5.0 — every district above that
value was recorded as 5.00001 (\$500,001), which `describe()` prints as 5.00. There is no starter code: write your answer in the cell below
and in new cells under it.

**Quantify what censoring does.**

1. How many districts sit at the ceiling?
2. Refit Step 6's model excluding them. What happens to the coefficients, and what happens to test
   RMSE?
3. Excluding them is not obviously right. Say what it assumes, and who the resulting model is now a
   model OF.
4. Reflect: this is a measurement decision of the kind Chapter 1 described, arriving in the middle
   of a modelling problem.

In [ ]:
# YOUR TASK — take-home Challenge: write your code here, and in new cells below

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Fitted the same OLS line three ways (the closed form, statsmodels
  and scikit-learn) and got the same intercept and slope from all three
* Added up to 100 columns of pure noise to a regression and watched
  R-squared rise every time, while adjusted R-squared did not
* Split the data before fitting, and scored an eight-feature model on
  the rows it was fitted to and on test rows it never saw
* Used one specification for two jobs: a coefficient with its 95%
  interval, and a test RMSE
* Wrote a small function, holdout_rmse(), that scores any list of
  predictors on the same test rows, and used it on median income alone
* Key findings: slope [YOUR VALUE]; R-squared [YOUR VALUE] with no
  noise and [YOUR VALUE] with 100 noise columns; train RMSE
  [YOUR VALUE] and test RMSE [YOUR VALUE]; AveRooms coefficient
  [YOUR VALUE], 95% CI [YOUR VALUE]; test RMSE with median income
  alone [YOUR VALUE]
* (Only if I did the take-home Challenge) Dropping the districts at
  the 5.0 ceiling: [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: OLS as a Line of Best Fit: Estimation versus Prediction
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab12-ols-prediction`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab12-ols-prediction`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab12-ols-prediction`, branch `main`,
   commit message `Lab 12 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.